# 🔬 DermAI v3 — HAM10000 + ISIC 2019 Combined Training

## Why this is better than v1 (75% overall)

| Problem in v1 | Fix in v3 |
|---|---|
| Only HAM10000 (~10k images) | **HAM10000 + ISIC 2019 (~35k images)** |
| mel accuracy 57% | More mel images from ISIC 2019 |
| akiec accuracy 52% | More akiec images from ISIC 2019 |
| Oversampling to 1500 | Oversampling to **3000** |
| EfficientNetB3 | **EfficientNetB4** (stronger) |
| Equal class weights | **Boosted weights** for rare/critical classes |
| 10+20 epochs | **15+30 epochs** |
| 30 unfrozen layers | **50 unfrozen layers** |

## Expected accuracy after v3
| Class | v1 | v3 Target |
|-------|-----|----------|
| mel | 57.4% | **~75%** |
| akiec | 52.9% | **~72%** |
| df | 55.2% | **~70%** |
| bkl | 57.7% | **~72%** |
| vasc | 67.9% | **~78%** |
| bcc | 74.5% | **~82%** |
| nv | 82.4% | **~88%** |
| **Overall** | **75%** | **~82%** |

> ⚠️ Set runtime to **GPU → T4** before running

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

## Step 2: Install Dependencies

In [ ]:
!pip install -q kaggle tensorflow scikit-learn pandas matplotlib seaborn Pillow
print('Done.')

## Step 3: Set Paths for HAM10000 (already on Drive)

In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/training model'

# ── HAM10000 paths (already uploaded) ─────────────────────────────────────
HAM_CSV  = os.path.join(DRIVE_BASE, 'HAM10000_metadata.csv')
HAM_DIRS = [
    os.path.join(DRIVE_BASE, 'HAM10000_images_part_1'),
    os.path.join(DRIVE_BASE, 'HAM10000_images_part_2'),
]

# ── ISIC 2019 will be downloaded to Colab local storage ───────────────────
ISIC_DIR = '/content/isic2019'
ISIC_CSV = '/content/isic2019/ISIC_2019_Training_GroundTruth.csv'
ISIC_IMG = '/content/isic2019/ISIC_2019_Training_Input'

# ── Output paths (saved to Drive) ─────────────────────────────────────────
MODEL_PATH   = os.path.join(DRIVE_BASE, 'dermai_model_v3.h5')
METADATA_OUT = os.path.join(DRIVE_BASE, 'dermai_class_metadata.json')

# Verify HAM10000
print('HAM10000 CSV exists:', os.path.exists(HAM_CSV))
for d in HAM_DIRS:
    exists = os.path.isdir(d)
    count  = len(os.listdir(d)) if exists else 0
    print(f'  {os.path.basename(d)}: exists={exists}, images={count}')

## Step 4: Download ISIC 2019 Dataset

ISIC 2019 has ~25,000 dermoscopy images — same 7 disease types as HAM10000.  
It's free and publicly available on Kaggle.

**Two ways to get it:**
- Option A: Use Kaggle API (if your key works)
- Option B: Upload manually to Drive like you did with HAM10000

In [ ]:
# ══════════════════════════════════════════════════════════════
# OPTION A: Kaggle API download (try this first)
# ══════════════════════════════════════════════════════════════
import json, os

# Paste your Kaggle credentials
KAGGLE_USERNAME = 'your_kaggle_username'   # ← change this
KAGGLE_KEY      = 'your_kaggle_api_key'    # ← change this

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download ISIC 2019 — correct dataset URL
os.makedirs(ISIC_DIR, exist_ok=True)
!kaggle datasets download -d andrewmvd/isic-2019 -p {ISIC_DIR}

!unzip -q {ISIC_DIR}/isic-2019.zip -d {ISIC_DIR} 2>/dev/null || true
print('ISIC 2019 download attempted.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# OPTION B: If Kaggle API doesn't work — upload ISIC to Drive
# then set this path instead:
# ══════════════════════════════════════════════════════════════

# If you uploaded ISIC 2019 to Google Drive, set USE_DRIVE_ISIC = True
# and update the path below
USE_DRIVE_ISIC = False   # ← set to True if using Drive

if USE_DRIVE_ISIC:
    ISIC_CSV = os.path.join(DRIVE_BASE, 'ISIC_2019_Training_GroundTruth.csv')
    ISIC_IMG = os.path.join(DRIVE_BASE, 'ISIC_2019_Training_Input')
    print('Using ISIC 2019 from Drive.')
else:
    print('Using ISIC 2019 from Kaggle download.')

# Check what we have
print('ISIC CSV exists:', os.path.exists(ISIC_CSV))
print('ISIC IMG dir exists:', os.path.isdir(ISIC_IMG))
if os.path.isdir(ISIC_IMG):
    print('ISIC images found:', len(os.listdir(ISIC_IMG)))

## Step 5: Load & Combine Both Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from glob import glob

# ── Load HAM10000 ──────────────────────────────────────────────────────────
df_ham = pd.read_csv(HAM_CSV)
print(f'HAM10000: {len(df_ham)} rows')

ham_paths = []
for d in HAM_DIRS:
    if os.path.isdir(d):
        found = glob(os.path.join(d, '*.jpg'))
        ham_paths.extend(found)

ham_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in ham_paths}
df_ham['path'] = df_ham['image_id'].map(ham_dict)
df_ham = df_ham.dropna(subset=['path'])[['image_id', 'dx', 'path']].reset_index(drop=True)
df_ham['source'] = 'ham10000'
print(f'HAM10000 matched: {len(df_ham)}')
print(df_ham['dx'].value_counts())

In [ ]:
# ── Load ISIC 2019 ─────────────────────────────────────────────────────────
# ISIC 2019 uses one-hot encoded columns instead of a single 'dx' column
# Columns: image, MEL, NV, BCC, AK, BKL, DF, VASC, SCC, UNK

ISIC_TO_HAM = {
    'MEL':  'mel',
    'NV':   'nv',
    'BCC':  'bcc',
    'AK':   'akiec',   # Actinic Keratosis
    'BKL':  'bkl',
    'DF':   'df',
    'VASC': 'vasc',
    # SCC and UNK are ignored — not in HAM10000
}

df_isic_combined = pd.DataFrame()

if os.path.exists(ISIC_CSV):
    df_isic_raw = pd.read_csv(ISIC_CSV)
    print(f'ISIC 2019 raw: {len(df_isic_raw)} rows')
    print('Columns:', df_isic_raw.columns.tolist())

    # Convert one-hot to label
    label_cols = [c for c in ISIC_TO_HAM.keys() if c in df_isic_raw.columns]
    df_isic_raw['dx_isic'] = df_isic_raw[label_cols].idxmax(axis=1)
    df_isic_raw['dx'] = df_isic_raw['dx_isic'].map(ISIC_TO_HAM)
    df_isic_raw = df_isic_raw.dropna(subset=['dx'])  # drop SCC/UNK

    # Map image paths
    isic_paths = glob(os.path.join(ISIC_IMG, '*.jpg'))
    isic_dict  = {os.path.splitext(os.path.basename(p))[0]: p for p in isic_paths}

    img_col = 'image' if 'image' in df_isic_raw.columns else df_isic_raw.columns[0]
    df_isic_raw['path'] = df_isic_raw[img_col].map(isic_dict)
    df_isic_raw = df_isic_raw.dropna(subset=['path'])

    df_isic_combined = df_isic_raw[['dx', 'path']].copy()
    df_isic_combined['image_id'] = df_isic_raw[img_col].values
    df_isic_combined['source']   = 'isic2019'

    print(f'ISIC 2019 matched: {len(df_isic_combined)}')
    print(df_isic_combined['dx'].value_counts())
else:
    print('ISIC 2019 not found — training on HAM10000 only.')
    print('Go back to Step 4 and either fix the Kaggle download or upload ISIC to Drive.')

In [ ]:
# ── Combine both datasets ──────────────────────────────────────────────────
if len(df_isic_combined) > 0:
    df = pd.concat([df_ham, df_isic_combined], ignore_index=True)
    print(f'Combined dataset: {len(df)} images')
else:
    df = df_ham.copy()
    print(f'Using HAM10000 only: {len(df)} images')

print('\nFinal class distribution:')
print(df['dx'].value_counts())

# Plot
plt.figure(figsize=(10, 4))
ax = df['dx'].value_counts().plot(kind='bar', color=plt.cm.tab10.colors[:7])
plt.title(f'Combined Dataset Distribution ({len(df)} images)', fontsize=13)
plt.xlabel('Disease Class')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Step 6: Labels + Stratified Split

In [ ]:
from sklearn.model_selection import train_test_split

CLASS_NAMES = sorted(df['dx'].unique().tolist())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

CLASS_DISPLAY_NAMES = {
    'akiec': 'Actinic Keratosis',
    'bcc':   'Basal Cell Carcinoma',
    'bkl':   'Benign Keratosis',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevi',
    'vasc':  'Vascular Lesion',
}

df['label'] = df['dx'].map(CLASS_TO_IDX)
print('Mapping:', CLASS_TO_IDX)

# Stratified 80/20 split
# Use lesion_id-aware split only for HAM (ISIC doesn't have lesion_id)
df_ham_only   = df[df['source'] == 'ham10000'].copy()
df_isic_only  = df[df['source'] == 'isic2019'].copy() if 'isic2019' in df['source'].values else pd.DataFrame()

# Split HAM with lesion-awareness if possible
if 'lesion_id' in df_ham_only.columns:
    unique_lesions = df_ham_only.groupby('lesion_id').first().reset_index()
    train_ids, val_ids = train_test_split(
        unique_lesions['lesion_id'], test_size=0.2,
        stratify=unique_lesions['dx'], random_state=42
    )
    ham_train = df_ham_only[df_ham_only['lesion_id'].isin(train_ids)]
    ham_val   = df_ham_only[df_ham_only['lesion_id'].isin(val_ids)]
else:
    ham_train, ham_val = train_test_split(
        df_ham_only, test_size=0.2, stratify=df_ham_only['dx'], random_state=42
    )

# Split ISIC normally
if len(df_isic_only) > 0:
    isic_train, isic_val = train_test_split(
        df_isic_only, test_size=0.2, stratify=df_isic_only['dx'], random_state=42
    )
    df_train = pd.concat([ham_train, isic_train]).reset_index(drop=True)
    df_val   = pd.concat([ham_val,   isic_val  ]).reset_index(drop=True)
else:
    df_train = ham_train.reset_index(drop=True)
    df_val   = ham_val.reset_index(drop=True)

print(f'Train: {len(df_train)} | Val: {len(df_val)}')
print('Train distribution:')
print(df_train['dx'].value_counts())

## Step 7: Oversample to 3000 per class + Boosted Class Weights

In [ ]:
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight

# Oversample every class to 3000
TARGET_PER_CLASS = 3000
balanced_parts = []
for cls in CLASS_NAMES:
    cls_df = df_train[df_train['dx'] == cls]
    cls_df = resample(cls_df, replace=True,
                      n_samples=TARGET_PER_CLASS, random_state=42)
    balanced_parts.append(cls_df)

df_train_bal = pd.concat(balanced_parts).sample(
    frac=1, random_state=42).reset_index(drop=True)

print('Balanced distribution (3000 each):')
print(df_train_bal['dx'].value_counts())
print(f'Total: {len(df_train_bal)}')

In [ ]:
# Boosted class weights for critical/rare classes
cw_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASS_NAMES)),
    y=df_train['label'].values
)

BOOST = {
    'mel':   2.0,    # cancer — most critical
    'akiec': 2.0,    # pre-cancer — critical
    'df':    2.5,    # very rare
    'vasc':  2.5,    # very rare
    'bkl':   1.5,    # low
    'bcc':   1.2,    # acceptable
    'nv':    0.5,    # dominant class — reduce weight
}

class_weights = {}
for i, cls in enumerate(CLASS_NAMES):
    class_weights[i] = cw_arr[i] * BOOST.get(cls, 1.0)

print('Class weights:')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  {cls:<8}  {class_weights[i]:.3f}')

## Step 8: Data Pipeline with Strong Augmentation

In [ ]:
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', len(tf.config.list_physical_devices('GPU')) > 0)

IMG_SIZE        = 256
BATCH_SIZE      = 32
EPOCHS_FROZEN   = 15
EPOCHS_FINETUNE = 30
LR_FROZEN       = 1e-3
LR_FINETUNE     = 5e-5
NUM_CLASSES     = len(CLASS_NAMES)
AUTOTUNE        = tf.data.AUTOTUNE

In [ ]:
def parse_image(path, label):
    img   = tf.io.read_file(path)
    img   = tf.image.decode_jpeg(img, channels=3)
    img   = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img   = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def augment(img, label):
    # Geometric
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    k   = tf.random.uniform([], 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    # Random zoom crop
    crop_frac = tf.random.uniform([], 0.75, 1.0)
    h = tf.cast(tf.cast(IMG_SIZE, tf.float32) * crop_frac, tf.int32)
    img = tf.image.random_crop(img, [h, h, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    # Colour jitter
    img = tf.image.random_brightness(img, max_delta=0.3)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    img = tf.image.random_hue(img, max_delta=0.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def build_dataset(dataframe, augment_data=False, shuffle=False):
    paths  = dataframe['path'].values
    labels = dataframe['label'].values.astype(int)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=42)
    ds = ds.map(parse_image, num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(df_train_bal, augment_data=True,  shuffle=True)
val_ds   = build_dataset(df_val,       augment_data=False, shuffle=False)

print(f'Train batches: {len(train_ds)} | Val batches: {len(val_ds)}')

## Step 9: Build Model — EfficientNetB4

In [ ]:
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB4

def build_model(num_classes, img_size=IMG_SIZE):
    inputs = layers.Input(shape=(img_size, img_size, 3))
    x      = layers.Rescaling(scale=255.0)(inputs)

    base = EfficientNetB4(
        include_top=False,
        weights='imagenet',
        input_tensor=x,
    )
    base.trainable = False  # frozen in Phase 1

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(1024, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs), base

model, base_model = build_model(NUM_CLASSES)
print(f'Total params: {model.count_params():,}')

## Step 10: Phase 1 — Train Head Only (15 epochs)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FROZEN),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_acc')]
)

cb_p1 = [
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1),
]

print('=' * 60)
print('PHASE 1 — Head only, 15 epochs')
print('=' * 60)
h1 = model.fit(train_ds, epochs=EPOCHS_FROZEN,
               validation_data=val_ds,
               class_weight=class_weights,
               callbacks=cb_p1, verbose=1)

## Step 11: Phase 2 — Fine-tune Top 50 Layers (30 epochs)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

print(f'Trainable base layers: {sum(1 for l in base_model.layers if l.trainable)} / {len(base_model.layers)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FINETUNE),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_acc')]
)

cb_p2 = [
    ModelCheckpoint(MODEL_PATH, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=4, min_lr=1e-9, verbose=1),
]

print('=' * 60)
print('PHASE 2 — Fine-tune top 50 layers, 30 epochs')
print('=' * 60)
h2 = model.fit(train_ds, epochs=EPOCHS_FINETUNE,
               validation_data=val_ds,
               class_weight=class_weights,
               callbacks=cb_p2, verbose=1)

## Step 12: Training Curves

In [ ]:
def merge(h1, h2):
    return {k: h1.history[k] + h2.history.get(k, []) for k in h1.history}

hist = merge(h1, h2)
p1   = len(h1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(hist['accuracy'],     label='Train', color='steelblue')
axes[0].plot(hist['val_accuracy'], label='Val',   color='orange')
axes[0].axvline(p1-1, color='gray', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist['loss'],     label='Train', color='steelblue')
axes[1].plot(hist['val_loss'], label='Val',   color='orange')
axes[1].axvline(p1-1, color='gray', linestyle='--')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('DermAI v3 — HAM10000 + ISIC 2019', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 13: Evaluation vs v1 Baseline

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(tf.argmax(labels, axis=1).numpy())
    y_pred.extend(tf.argmax(preds,  axis=1).numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt='.2f',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Blues', linewidths=0.5)
plt.title('Confusion Matrix v3', fontsize=13)
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Compare with v1
v1 = {'nv':0.824,'bcc':0.745,'vasc':0.679,'bkl':0.577,'mel':0.574,'df':0.552,'akiec':0.529}
print(f'\n{"Class":<8} {"v1":>7} {"v3":>7} {"Change":>8}')
print('-' * 34)
for cls, acc in zip(CLASS_NAMES, cm.diagonal() / cm.sum(axis=1)):
    old   = v1.get(cls, 0)
    delta = acc - old
    arrow = '↑' if delta > 0 else '↓'
    print(f'{cls:<8} {old*100:>6.1f}%  {acc*100:>6.1f}%  {arrow}{abs(delta)*100:.1f}%')

## Step 14: Save Model & Metadata to Drive

In [ ]:
import json

model.save(MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

metadata = {
    'class_names':         CLASS_NAMES,
    'class_display_names': CLASS_DISPLAY_NAMES,
    'class_to_idx':        CLASS_TO_IDX,
    'idx_to_class':        {str(k): v for k, v in IDX_TO_CLASS.items()},
    'img_size':            IMG_SIZE,
    'num_classes':         NUM_CLASSES,
    'risk_mapping': {
        'mel':   'High Risk',
        'bcc':   'High Risk',
        'akiec': 'High Risk',
        'bkl':   'Low Risk',
        'df':    'Low Risk',
        'nv':    'Low Risk',
        'vasc':  'Medium Risk',
    },
    'explanation': {
        'mel':   'The lesion exhibits asymmetric borders, color variation, and irregular shape — key indicators of Melanoma. Immediate dermatologist consultation is strongly recommended.',
        'bcc':   'Characteristics suggest Basal Cell Carcinoma — the most common form of skin cancer. Early treatment is highly effective. Please consult a dermatologist.',
        'akiec': 'Findings are consistent with Actinic Keratosis or early intraepithelial carcinoma. These are considered pre-cancerous and should be monitored closely.',
        'bkl':   'The pattern is consistent with a Benign Keratosis-like lesion. Generally harmless, but monitoring is advised.',
        'df':    'Lesion is consistent with Dermatofibroma, a benign fibrous nodule. Usually harmless; excision is only necessary if symptomatic.',
        'nv':    'The lesion appears to be a benign Melanocytic Nevus (mole). Regular monitoring is advised.',
        'vasc':  'Findings suggest a Vascular Lesion. These are typically benign but should be evaluated by a dermatologist.',
    },
    'suggestions': {
        'mel':   ['Consult a dermatologist immediately.', 'Avoid sun exposure.', 'Consider biopsy for confirmation.'],
        'bcc':   ['Schedule dermatologist appointment.', 'Avoid prolonged UV exposure.', 'Surgical removal may be required.'],
        'akiec': ['Consult a dermatologist promptly.', 'Use SPF 50+ sunscreen daily.', 'Cryotherapy or topical treatment may be recommended.'],
        'bkl':   ['Monitor for any changes.', 'Annual skin check recommended.', 'No immediate treatment needed.'],
        'df':    ['Usually no treatment needed.', 'Monitor for size or color changes.', 'See a dermatologist if it becomes painful.'],
        'nv':    ['Monitor for changes in size or color.', 'Annual skin check recommended.', 'Use SPF 50+ sunscreen.'],
        'vasc':  ['Consult a dermatologist for evaluation.', 'Laser treatment may be an option.', 'Monitor for any changes in size or bleeding.'],
    }
}

with open(METADATA_OUT, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved → {METADATA_OUT}')
print('\n✅ Files saved to Drive/training model/')
print('   dermai_model_v3.h5')
print('   dermai_class_metadata.json')

## After Training — Steps to Deploy

1. Go to **Google Drive → training model folder**
2. Download `dermai_model_v3.h5`
3. Rename it to `dermai_model.h5`
4. Replace the old `dermai_model.h5` in your project folder
5. Download `dermai_class_metadata.json` and replace it too
6. Restart Flask — done ✅

## How to get ISIC 2019 if Kaggle API doesn't work

1. Go to **[kaggle.com/datasets/andrewmvd/isic-2019](https://www.kaggle.com/datasets/andrewmvd/isic-2019)**
2. Click **Download** (you need to be logged in to Kaggle)
3. It downloads `isic-2019.zip` (~9 GB)
4. Extract it — you'll get:
   - `ISIC_2019_Training_Input/` folder with images
   - `ISIC_2019_Training_GroundTruth.csv`
5. Upload both to Google Drive under `My Drive/training model/`
6. In Step 4 of this notebook, set `USE_DRIVE_ISIC = True`